# Merkle Data Engineer Case Study

## 1. Solution overview

Two S3 CSV sources are processed through three Delta layers: raw ingestion, typed and flattened datasets, and a yearly item-view mart.

**S3 CSV → Layer 1 raw Delta tables → Layer 2 cleaned and flattened Delta tables → Layer 3 `top_item` mart → final reconciliation checks**

Writes use full overwrite so reruns replace each table rather than append duplicate data. Critical checks stop execution before event pivoting or mart publication.


## 2. Setup and source configuration

Run on Databricks with access to the S3 sources and permission to create schemas and Delta tables in the current catalog. Timestamps use the source format `yyyy-MM-dd HH:mm:ss` and the Spark session timezone.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
ITEM_PATH = "s3a://merkle-de-interview-case-study/de/item.csv"
EVENT_PATH = "s3a://merkle-de-interview-case-study/de/event.csv"

item_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ITEM_PATH)
)

event_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("quote", '"')
    .option("escape", '"')
    .csv(EVENT_PATH)
)

In [0]:
payload_schema = T.StructType([
    T.StructField("event_name", T.StringType(), True),
    T.StructField("platform", T.StringType(), True),
    T.StructField("parameter_name", T.StringType(), True),
    T.StructField("parameter_value", T.StringType(), True),
])

In [0]:
# Create schemas for the three logical data lake layers.
spark.sql("CREATE SCHEMA IF NOT EXISTS layer1")
spark.sql("CREATE SCHEMA IF NOT EXISTS layer2")
spark.sql("CREATE SCHEMA IF NOT EXISTS layer3")

DataFrame[]

## 3. Source data exploration

`item.csv` describes items. `event.csv` contains event identifiers, timestamps, user identifiers, and a JSON `event.payload` with event name, platform, parameter name, and parameter value.

Parameters are stored vertically: multiple source rows describe one logical event identified by `event_id`. The event-type/parameter combinations and rows per event below explain the pivot design. These are source parameter-row counts, not logical event counts; consistency is validated in Layer 2.


In [0]:
print(f"Item source rows: {item_raw.count():,}")
print(f"Event source parameter rows: {event_raw.count():,}")
item_raw.show(5, truncate=False)

Item source rows: 2,198
Event source parameter rows: 853,640
+-------------------+-----------+-------------------+------+-------------+-------------------------------+-----+
|adjective          |category   |created_at         |id    |modifier     |name                           |price|
+-------------------+-----------+-------------------+------+-------------+-------------------------------+-----+
|fuzzy              |contraption|2014-01-15 21:36:09|2512.0|carrying_case|fuzzy contraption carrying_case|150.0|
|NULL               |instrument |2013-05-14 05:20:50|482.0 |refill       |instrument refill              |35.2 |
|industrial-strength|module     |2014-02-04 19:28:32|2446.0|NULL         |industrial-strength module     |300.0|
|digital            |tool       |2013-02-25 12:23:18|1312.0|carrying_case|digital tool carrying_case     |16.5 |
|miniature          |device     |2013-08-05 17:20:45|3556.0|cleaner      |miniature device cleaner       |16.5 |
+-------------------+-----------+--

In [0]:
event_parsed = (
    event_raw
    .withColumn(
        "payload",
        F.from_json(F.col("`event.payload`"), payload_schema)
    )
    .select(
        "event_id",
        "event_time",
        "user_id",
        F.col("payload.event_name").alias("event_name"),
        F.col("payload.platform").alias("platform"),
        F.col("payload.parameter_name").alias("parameter_name"),
        F.col("payload.parameter_value").alias("parameter_value"),
    )
)

event_parsed.show(5, truncate=False)

print("Source rows by event type and parameter:")
(
    event_parsed
    .groupBy("event_name", "parameter_name")
    .count()
    .orderBy("event_name", "parameter_name")
    .show(truncate=False)
)

print("Rows per logical event_id:")
(
    event_parsed
    .groupBy("event_id")
    .count()
    .withColumnRenamed("count", "source_rows")
    .groupBy("source_rows")
    .agg(F.count("*").alias("logical_events"))
    .orderBy("source_rows")
    .show()
)

+--------------------------------+-------------------+--------+----------+--------+--------------+---------------+
|event_id                        |event_time         |user_id |event_name|platform|parameter_name|parameter_value|
+--------------------------------+-------------------+--------+----------+--------+--------------+---------------+
|b9de71c5c3cc4cd7a97e50b832106e5a|2017-06-26 11:23:39|178481.0|view_item |android |item_id       |3526           |
|23267713c9ea44419331731f50b6a8db|2017-06-27 10:46:39|178481.0|view_item |android |item_id       |1514           |
|1b7822fa7b854e01970218ae8f721fe0|2017-06-27 11:15:39|178481.0|view_item |android |item_id       |3712           |
|2a7a188a626841ac94befcc419f06af4|2016-10-05 20:43:10|154133.0|view_item |android |item_id       |3586           |
|631d657264cc4616a4528f759509b25d|2016-10-04 03:29:10|154133.0|view_item |android |item_id       |1061           |
+--------------------------------+-------------------+--------+----------+------

## 4. Layer 1 — Raw

`layer1.item` and `layer1.event` preserve the source columns and values as read from CSV, without business transformations. Schema inference is disabled so every source attribute remains STRING, including the original JSON payload. Parsing and type conversion belong to Layer 2.


In [0]:
assert all(isinstance(field.dataType, T.StringType) for field in item_raw.schema.fields), "Layer 1 item columns must all be STRING"
assert all(isinstance(field.dataType, T.StringType) for field in event_raw.schema.fields), "Layer 1 event columns must all be STRING"

# Persist the source datasets without transformations.
(
    item_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("layer1.item")
)

(
    event_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("layer1.event")
)

## 5. Layer 2 — Clean

Layer 2 uses snake_case names, appropriate data types, and one row per business entity or logical event.

### 5.1 Item

`id` becomes `item_id` (LONG). An exact DECIMAL intermediate supports source identifiers such as `2512.0` without floating-point precision loss. `created_at` becomes TIMESTAMP and `price` uses DECIMAL(12,2) for monetary values; descriptive fields remain STRING.


In [0]:
item_clean = (
    spark.table("layer1.item")
    .select(
        F.col("id").cast(T.DecimalType(38, 18)).cast("long").alias("item_id"),
        F.col("name"),
        F.col("category"),
        F.col("adjective"),
        F.col("modifier"),
        F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss").alias("created_at"),
        F.col("price").cast(T.DecimalType(12, 2)).alias("price")
    )
)

item_clean.printSchema()

root
 |-- item_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- adjective: string (nullable = true)
 |-- modifier: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)



In [0]:
(
    item_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("layer2.item")
)

### 5.2 Event

Rows sharing an `event_id` must have the same timestamp, user, event name, and platform. Non-null fields, valid payload parsing, expected parameter names, and consistent parameter values are checked before pivoting.

The pivot turns vertical parameters into columns, producing **one row per logical event (`event_id`)**. Repeated identical parameter values collapse to one value; conflicting values fail validation.

`event_id` remains STRING; user and item identifiers become LONG. `user_id` uses the same exact DECIMAL intermediate as item identifiers. Numeric test fields become INT, while event names, platforms, and referrers remain STRING. `event_time` becomes TIMESTAMP, with derived DATE `event_date` and INT `event_year`. Parameters not applicable to an event remain null.

The fact-like event table is physically partitioned by `event_year`: its low cardinality supports year-filtered analytical queries. High-cardinality event, user, and item IDs would create too many small partitions.


In [0]:
# Parse the JSON payload from the raw Layer 1 event table.
event_payload_parsed = (
    spark.table("layer1.event")
    .withColumn(
        "payload",
        F.from_json(F.col("`event.payload`"), payload_schema)
    )
)

assert event_payload_parsed.filter(
    F.col("`event.payload`").isNotNull() & F.col("payload").isNull()
).limit(1).count() == 0, "Non-null event payload failed JSON parsing"

event_payload = (
    event_payload_parsed
    .select(
        "event_id",
        "event_time",
        "user_id",
        F.col("payload.event_name").alias("event_name"),
        F.col("payload.platform").alias("platform"),
        F.col("payload.parameter_name").alias("parameter_name"),
        F.col("payload.parameter_value").alias("parameter_value")
    )
)

# Validate event grain and parameters before pivoting.
expected_parameters = [
    "item_id",
    "referrer",
    "test_id",
    "test_assignment",
    "viewed_user_id"
]

assert event_payload.filter(
    F.col("event_id").isNull() |
    F.col("event_time").isNull() |
    F.col("user_id").isNull() |
    F.col("event_name").isNull() |
    F.col("platform").isNull() |
    F.col("parameter_name").isNull() |
    F.col("parameter_value").isNull()
).limit(1).count() == 0, "Null event fields or parameters"

assert event_payload.filter(
    ~F.col("parameter_name").isin(expected_parameters)
).limit(1).count() == 0, "Unexpected event parameter names"

event_attribute_counts = (
    event_payload
    .groupBy("event_id")
    .agg(
        F.countDistinct("event_time").alias("event_time_values"),
        F.countDistinct("user_id").alias("user_id_values"),
        F.countDistinct("event_name").alias("event_name_values"),
        F.countDistinct("platform").alias("platform_values")
    )
)

assert event_attribute_counts.filter(
    (F.col("event_time_values") != 1) |
    (F.col("user_id_values") != 1) |
    (F.col("event_name_values") != 1) |
    (F.col("platform_values") != 1)
).limit(1).count() == 0, "Inconsistent attributes for the same event_id"

event_parameter_counts = (
    event_payload
    .groupBy("event_id", "parameter_name")
    .agg(F.countDistinct("parameter_value").alias("parameter_values"))
)

assert event_parameter_counts.filter(
    F.col("parameter_values") > 1
).limit(1).count() == 0, "Conflicting values for the same event parameter"

In [0]:
# Flatten vertically stored event parameters into one row per logical event.
event_flat = (
    event_payload
    .groupBy(
        "event_id",
        "event_time",
        "user_id",
        "event_name",
        "platform"
    )
    .pivot(
        "parameter_name",
        expected_parameters
    )
    .agg(F.first("parameter_value", ignorenulls=True))
)

# Apply proper data types and derive useful time attributes.
event_clean = (
    event_flat
    .select(
        F.col("event_id"),
        F.to_timestamp(
            "event_time",
            "yyyy-MM-dd HH:mm:ss"
        ).alias("event_time"),
        F.col("user_id").cast(T.DecimalType(38, 18)).cast("long").alias("user_id"),
        F.col("event_name"),
        F.col("platform"),
        F.col("item_id").cast("long").alias("item_id"),
        F.col("referrer"),
        F.col("test_id").cast("int").alias("test_id"),
        F.col("test_assignment").cast("int").alias("test_assignment"),
        F.col("viewed_user_id").cast("long").alias("viewed_user_id")
    )
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("event_year", F.year("event_time"))
)

event_clean.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- item_id: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- test_id: integer (nullable = true)
 |-- test_assignment: integer (nullable = true)
 |-- viewed_user_id: long (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_year: integer (nullable = true)



In [0]:
print("Required parameter validation:")

invalid_view_item = (
    event_clean
    .filter(
        (F.col("event_name") == "view_item") &
        (
            F.col("item_id").isNull() |
            F.col("referrer").isNull()
        )
    )
    .count()
)

invalid_test_assignment = (
    event_clean
    .filter(
        (F.col("event_name") == "test_assignment") &
        (
            F.col("test_id").isNull() |
            F.col("test_assignment").isNull()
        )
    )
    .count()
)

invalid_profile_view = (
    event_clean
    .filter(
        (F.col("event_name") == "view_user_profile") &
        F.col("viewed_user_id").isNull()
    )
    .count()
)

print(f"Invalid view_item events: {invalid_view_item}")
print(f"Invalid test_assignment events: {invalid_test_assignment}")
print(f"Invalid view_user_profile events: {invalid_profile_view}")

Required parameter validation:
Invalid view_item events: 0
Invalid test_assignment events: 0
Invalid view_user_profile events: 0


In [0]:
(
    event_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("event_year")
    .saveAsTable("layer2.event")
)

In [0]:
spark.sql("DESCRIBE DETAIL layer2.event").select(
    "format",
    "numFiles",
    "partitionColumns"
).show(truncate=False)

+------+--------+----------------+
|format|numFiles|partitionColumns|
+------+--------+----------------+
|delta |6       |[event_year]    |
+------+--------+----------------+



## 6. Layer 3 — `top_item` data mart

The mart contains **one row per `(item_id, year)` with at least one `view_item` event**. Item keys must be unique and non-null; every view must have an item, year, and platform, and reference an existing item.

| Attribute | Meaning |
| --- | --- |
| `item_id` | Item identifier (LONG). |
| `item_name` | Item name from `layer2.item` (STRING). |
| `year` | Calendar year of the views (INT). |
| `total_views` | Number of logical `view_item` events for the item/year (LONG). |
| `item_rank` | Rank within the year by descending `total_views` (INT). |
| `most_used_platform` | Platform with the most views for the item/year (STRING). |

`RANK()` gives equally viewed items the same rank, with gaps after ties (for example, 1, 1, 3). Platform usage is counted per item/year/platform; `row_number()` selects the highest count, with ascending platform name resolving ties deterministically.

The small aggregated mart is not physically partitioned: its item/year grain already reduces the event volume, and extra partitions would add unnecessary file overhead.


In [0]:
item_keys = spark.table("layer2.item").select("item_id")

assert item_keys.filter(
    F.col("item_id").isNull()
).limit(1).count() == 0, "Null item IDs in layer2.item"

assert item_keys.groupBy("item_id").count().filter(
    F.col("count") > 1
).limit(1).count() == 0, "Duplicate item IDs in layer2.item"

view_items = (
    spark.table("layer2.event")
    .filter(F.col("event_name") == "view_item")
)

assert view_items.filter(
    F.col("item_id").isNull() |
    F.col("event_year").isNull() |
    F.col("platform").isNull()
).limit(1).count() == 0, "Null item, year, or platform in view_item events"

unknown_items = (
    view_items
    .select("item_id")
    .distinct()
    .join(
        item_keys,
        on="item_id",
        how="left_anti"
    )
)

assert unknown_items.limit(1).count() == 0, "Viewed item IDs missing from layer2.item"

In [0]:
view_events = (
    spark.table("layer2.event")
    .filter(F.col("event_name") == "view_item")
    .select(
        "event_id",
        "item_id",
        "event_year",
        "platform"
    )
)

# Total number of item views per year.
item_year_views = (
    view_events
    .groupBy("item_id", "event_year")
    .agg(
        F.count("*").alias("total_views")
    )
)

# Rank items within each year by number of views.
year_rank_window = (
    Window
    .partitionBy("event_year")
    .orderBy(F.desc("total_views"))
)

item_year_ranked = (
    item_year_views
    .withColumn(
        "item_rank",
        F.rank().over(year_rank_window)
    )
)

# Count views by platform for every item and year.
platform_counts = (
    view_events
    .groupBy(
        "item_id",
        "event_year",
        "platform"
    )
    .agg(
        F.count("*").alias("platform_views")
    )
)

# Select the most frequently used platform.
# Alphabetical platform ordering provides deterministic tie handling.
platform_rank_window = (
    Window
    .partitionBy("item_id", "event_year")
    .orderBy(
        F.desc("platform_views"),
        F.asc("platform")
    )
)

most_used_platform = (
    platform_counts
    .withColumn(
        "platform_rank",
        F.row_number().over(platform_rank_window)
    )
    .filter(F.col("platform_rank") == 1)
    .select(
        "item_id",
        "event_year",
        F.col("platform").alias("most_used_platform")
    )
)

top_item = (
    item_year_ranked
    .join(
        most_used_platform,
        on=["item_id", "event_year"],
        how="inner"
    )
    .join(
        spark.table("layer2.item").select(
            "item_id",
            F.col("name").alias("item_name")
        ),
        on="item_id",
        how="left"
    )
    .select(
        "item_id",
        "item_name",
        F.col("event_year").alias("year"),
        "total_views",
        "item_rank",
        "most_used_platform"
    )
)


## 7. Final validation and result

Before publication, assertions check unique, non-null item/year keys and reconcile total views against Layer 2, both overall and for every year. Together with the input checks, these guard against lost or multiplied views and invalid item references. `layer3.top_item` is written only after all critical checks pass.

The final output shows yearly reconciliation, the persisted schema, and items ranked in the top five positions per year. Shared ranks can return more than five items.


In [0]:
print(f"Rows: {top_item.count():,}")

duplicate_rows = (
    top_item
    .groupBy("item_id", "year")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_key_rows = (
    top_item
    .filter(
        F.col("item_id").isNull() |
        F.col("year").isNull()
    )
    .count()
)

mart_total_views = (
    top_item
    .agg(F.sum("total_views").alias("views"))
    .first()["views"] or 0
)

source_total_views = (
    spark.table("layer2.event")
    .filter(F.col("event_name") == "view_item")
    .count()
)

print(f"Duplicate item/year rows: {duplicate_rows}")
print(f"Null business keys: {null_key_rows}")
print(f"Views represented in mart: {mart_total_views:,}")
print(f"Source view_item events: {source_total_views:,}")

assert duplicate_rows == 0, "Duplicate item/year rows in top_item"
assert null_key_rows == 0, "Null business keys in top_item"
assert mart_total_views == source_total_views, "Overall view totals do not match"

mart_yearly_views = (
    top_item
    .groupBy("year")
    .agg(F.sum("total_views").alias("mart_views"))
)

source_yearly_views = (
    view_events
    .groupBy("event_year")
    .agg(F.count("*").alias("source_views"))
    .withColumnRenamed("event_year", "year")
)

yearly_view_mismatches = (
    mart_yearly_views
    .join(source_yearly_views, on="year", how="full")
    .filter(
        F.col("mart_views").isNull() |
        F.col("source_views").isNull() |
        (F.col("mart_views") != F.col("source_views"))
    )
)

assert yearly_view_mismatches.limit(1).count() == 0, "Yearly view totals do not match"

Rows: 13,093
Duplicate item/year rows: 0
Null business keys: 0
Views represented in mart: 262,786
Source view_item events: 262,786


In [0]:
(
    top_item.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("layer3.top_item")
)

In [0]:
print("Yearly view reconciliation:")
(
    mart_yearly_views
    .join(source_yearly_views, on="year", how="full")
    .orderBy("year")
    .show(truncate=False)
)

print("Published top_item schema:")
spark.table("layer3.top_item").printSchema()

print("Items ranked in the top five positions per year:")
(
    spark.table("layer3.top_item")
    .filter(F.col("item_rank") <= 5)
    .orderBy("year", "item_rank", "item_id")
    .show(100, truncate=False)
)

Yearly view reconciliation:
+----+----------+------------+
|year|mart_views|source_views|
+----+----------+------------+
|2013|7448      |7448        |
|2014|32082     |32082       |
|2015|49462     |49462       |
|2016|63642     |63642       |
|2017|76314     |76314       |
|2018|33838     |33838       |
+----+----------+------------+

Published top_item schema:
root
 |-- item_id: long (nullable = true)
 |-- item_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- total_views: long (nullable = true)
 |-- item_rank: integer (nullable = true)
 |-- most_used_platform: string (nullable = true)

Items ranked in the top five positions per year:
+-------+------------------------------------------+----+-----------+---------+------------------+
|item_id|item_name                                 |year|total_views|item_rank|most_used_platform|
+-------+------------------------------------------+----+-----------+---------+------------------+
|2086   |organic device           